# 👑 MỐC 3: BẢN HOÀN CHỈNH VÔ ĐỊCH (CHAMPION SOTA SUITE - 5-FOLD ENSEMBLE SUITE)
### Đề tài: *Harmonizing Local Inductive Bias and Global Contextual Reasoning: A Dual-Branch ConvNeXt-Swin Transformer Architecture with Gender-Modulated Cross-Attention for Pediatric Bone Age Assessment*

---
## 🎯 Mục Tiêu Khoa Học & Bộ 3 Đòn Bẩy Kỹ Thuật (Champion SOTA Boosts):
1. **Kiến trúc Spatial Token-Level Cross-Attention ($49 	imes 144$):**
   - **ConvNeXt-Tiny (384×384):** Trích xuất $12 	imes 12 = 144$ micro tokens biểu diễn đĩa sụn tăng trưởng và viền vôi hóa xương bè.
   - **Swin-Tiny (224×224):** Trích xuất $7 	imes 7 = 49$ macro tokens đại diện cho 49 vùng giải phẫu theo chuẩn Tanner-Whitehouse (TW3).
   - Mỗi token vĩ mô của Swin truy vấn 144 vùng vi mô của ConvNeXt, giải quyết triệt để lỗi suy biến ma trận $1 	imes 1$ của mô hình cũ.
2. **Giải phóng Gradient ở Vùng Sai Số Nhỏ (Smooth L1 $\beta = 0.1$ trên Z-Score):**
   - Thu hẹp vùng bậc hai $L_2$ từ 41.2 tháng xuống chỉ còn **4.1 tháng** ($eta = 0.1 	imes \sigma$).
   - Duy trì gradient không đổi $\pm 1.0$ ở mọi sai số $> 4.1$ tháng, giúp mô hình bứt phá khỏi mức sàn 6.9m mà các hàm loss cũ bị nghẽn.
3. **Chiến Lược Vũ Khí Vô Địch RSNA: 5-Fold Stratified Cross-Validation & Ensemble Blending:**
   - 10,088 mẫu train được chia thành 5 folds phân tầng theo tuổi và giới tính. Cố định 1,262 mẫu Test độc lập.
   - Mỗi lượt chạy chỉ huấn luyện 1 Fold độc lập (`CONFIG['FOLD'] = 0..4`) trong ~75–80 phút, tuyệt đối an toàn hạn ngạch GPU.
   - **Cell Tự Động Soft-Voting Ensemble:** Nạp tất cả các file dự đoán của các Fold đã chạy và lấy trung bình dự đoán trên tập Test độc lập, triệt tiêu phương sai nhiễu ngẫu nhiên để kéo MAE xuống vùng **4.26 – 4.60 tháng**!


In [ ]:
# Cài đặt thư viện phụ trợ (timm, albumentations)
!pip install -q timm albumentations tabulate


In [ ]:
# ==============================================================================
# BƯỚC 1: CẤU HÌNH SIÊU THAM SỐ TẬP TRUNG & MÔI TRƯỜNG THỰC THI
# ==============================================================================
import os
import sys
import time
import math
import json
import random
from pathlib import Path

# Triệt tiêu vĩnh viễn nguy cơ Deadlock đa luồng giữa OpenCV và PyTorch DataLoader
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import albumentations as A
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm

# Lọc bỏ cảnh báo dọn dẹp bộ nhớ vô hại của Python 3.13 DataLoader worker __del__
def silence_dataloader_gc(unraisable):
    if unraisable.exc_type is AssertionError or "dataloader" in str(unraisable.object).lower():
        return
    sys.__unraisablehook__(unraisable)
sys.unraisablehook = silence_dataloader_gc

# Thiết lập hiển thị đồ họa
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# BẢNG CẤU HÌNH THÍ NGHIỆM EXP 5 (CHAMPION SOTA SUITE)
CONFIG = {
    'EXPERIMENT': 'exp5_champion_dual_branch',
    'SEED': 42,
    'FOLD': 0,                  # Chỉ số Fold cần huấn luyện (Chọn 0, 1, 2, 3, hoặc 4)
    'NUM_FOLDS': 5,             # Tổng số Folds phân tầng
    'USE_ZSCORE': True,         # Chuẩn hóa Z-Score nhãn tuổi xương
    'USE_TTA': True,            # Bật Test-Time Augmentation khi kiểm định
    'CONVNEXT_MODEL': 'convnext_tiny.fb_in22k_ft_in1k_384',
    'SWIN_MODEL': 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
    'IMAGE_SIZE_CONV': 384,     # Kích thước nhánh vi mô
    'IMAGE_SIZE_SWIN': 224,     # Kích thước nhánh vĩ mô
    'EMBED_DIM': 512,
    'NUM_HEADS': 8,
    'DROPOUT': 0.3,
    'DROP_PATH_RATE': 0.2,      # Stochastic Depth chống overfit trên 10k ảnh
    'BATCH_SIZE': 16,
    'NUM_WORKERS': 2,           # 2 workers an toàn tuyệt đối tránh nghẽn shared memory /dev/shm
    'EPOCHS': 26,               # 26 epochs (2 Warmup + 24 Cosine) để chạm đáy hội tụ
    'WARMUP_EPOCHS': 2,
    'SMOOTH_L1_BETA': 0.1,      # Beta=0.1 (tương đương 4.1 tháng) duy trì gradient ±1 khi sai số < 8m
    'LR_HEAD_WARMUP': 1e-3,
    'LR_BACKBONE': 2.5e-5,      # Đẩy tốc độ hội tụ backbone lên 2.5e-5
    'LR_HEAD': 3e-4,
    'WEIGHT_DECAY': 0.05,       # Chuẩn AdamW cho Vision Transformers
    'GRADIENT_ACCUMULATION': 2, # Effective Batch Size = 32
    'USE_AMP': True,
    'CHECKPOINT_DIR': '/kaggle/working/checkpoints',
    'OUTPUT_DIR': '/kaggle/working/outputs',
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CONFIG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Môi trường phần cứng: {device} | PyTorch: {torch.__version__}", flush=True)
if torch.cuda.is_available():
    print(f"🚀 Tên GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB", flush=True)
sys.stdout.flush()

os.makedirs(CONFIG['CHECKPOINT_DIR'], exist_ok=True)
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)


---
## 📂 BƯỚC 2: TỰ ĐỘNG KẾT NỐI DATASET & PHÂN TẦNG 5-FOLD CHỐNG RÒ RỈ
- Cố định **1,262 mẫu Test độc lập** (`split == 'test'`) chưa từng xuất hiện trong quá trình huấn luyện.
- 10,088 mẫu còn lại được chia thành **5 Stratified Folds** đồng đều theo phân vị tuổi và giới tính sinh học.
- Tự động nạp Fold được chọn qua `CONFIG['FOLD']` (0 đến 4) và tính Z-Score chuẩn.


In [ ]:
# ==============================================================================
# BƯỚC 2: KẾT NỐI DỮ LIỆU & PHÂN CHIA 5-FOLD STRATIFIED K-FOLD
# ==============================================================================
DATASET_DIR = None
STRATIFIED_CSV_PATH = None
IMAGES_DIR = None

if Path('/kaggle/input').exists():
    for root, dirs, files in os.walk('/kaggle/input'):
        dirs[:] = [d for d in dirs if d.lower() not in ['images', 'image', 'boneage-training-dataset', 'boneage-test-dataset']]
        if 'train_stratified.csv' in files:
            STRATIFIED_CSV_PATH = Path(root) / 'train_stratified.csv'
            DATASET_DIR = Path(root)
            break

if STRATIFIED_CSV_PATH is None:
    if Path('/kaggle/working/rsna_processed_512/train_stratified.csv').exists():
        STRATIFIED_CSV_PATH = Path('/kaggle/working/rsna_processed_512/train_stratified.csv')
        DATASET_DIR = Path('/kaggle/working/rsna_processed_512')
    elif Path('./rsna_processed_512/train_stratified.csv').exists():
        STRATIFIED_CSV_PATH = Path('./rsna_processed_512/train_stratified.csv')
        DATASET_DIR = Path('./rsna_processed_512')

if STRATIFIED_CSV_PATH is None:
    raise FileNotFoundError("❌ CHƯA TÌM THẤY TỆP 'train_stratified.csv'! Hãy gán Dataset từ Notebook 01.")

IMAGES_DIR = DATASET_DIR / 'images'
df_all = pd.read_csv(STRATIFIED_CSV_PATH)

# Tách riêng tập Test độc lập (1,262 mẫu)
test_df = df_all[df_all['split'] == 'test'].reset_index(drop=True)
pool_df = df_all[df_all['split'] != 'test'].reset_index(drop=True)

# Tạo khóa phân tầng kết hợp (Age Quantile x Gender)
strata_keys = pool_df['age_quantile'].astype(str) + "_" + pool_df['male'].astype(str)
skf = StratifiedKFold(n_splits=CONFIG['NUM_FOLDS'], shuffle=True, random_state=CONFIG['SEED'])

pool_df['fold'] = -1
for f_idx, (train_indices, val_indices) in enumerate(skf.split(pool_df, strata_keys)):
    pool_df.loc[val_indices, 'fold'] = f_idx

# Lấy tập Train và Val cho Fold hiện tại
current_fold = CONFIG['FOLD']
train_df = pool_df[pool_df['fold'] != current_fold].reset_index(drop=True)
val_df = pool_df[pool_df['fold'] == current_fold].reset_index(drop=True)

# Thống kê Z-Score chuẩn tính riêng trên tập Train của Fold này
BONEAGE_MEAN = float(train_df['boneage'].mean())
BONEAGE_STD = float(train_df['boneage'].std())

print("=" * 60)
print(f"✅ ĐÃ PHÂN CHIA 5-FOLD STRATIFIED HOÀN TẤT (ĐANG CHỌN FOLD {current_fold}):")
print(f"• Thư mục ảnh:           {IMAGES_DIR}")
print(f"• Tập Huấn luyện (Train): {len(train_df):,} mẫu (4 Folds)")
print(f"• Tập Kiểm định (Val):   {len(val_df):,} mẫu (Fold {current_fold})")
print(f"• Tập Kiểm thử (Test):   {len(test_df):,} mẫu (Holdout Test cố định)")
print(f"• Thống kê Z-Score:      Mean (μ) = {BONEAGE_MEAN:.2f} tháng | Std (σ) = {BONEAGE_STD:.2f} tháng")
print("=" * 60)


---
## 📦 BƯỚC 3: MEDICAL AUGMENTATION PIPELINE & CHUẨN HÓA Z-SCORE
- Áp dụng các phép biến đổi nhẹ nhàng qua `albumentations` bảo toàn cấu trúc giải phẫu.
- Tích hợp CoarseDropout (Cutout) chống học vẹt trên nền X-quang.
- Tự động chuẩn hóa nhãn $y_{\text{norm}} = \frac{y - \mu}{\sigma}$ trong lúc nạp mẫu.


In [ ]:
# ==============================================================================
# BƯỚC 3: ĐỊNH NGHĨA AUGMENTATION VÀ DATASET CHUYÊN SÂU
# ==============================================================================

# Pipeline tăng cường dữ liệu y khoa chống học vẹt (Medical Affine + CoarseDropout)
train_transform_cv = A.Compose([
    A.Affine(scale=(0.92, 1.08), translate_percent=(-0.05, 0.05), rotate=(-10, 10), border_mode=0, fill=0, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.5),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(16, 36), hole_width_range=(16, 36), fill=0, p=0.5),
])

class RSNASotaDataset(Dataset):
    def __init__(self, df, img_dir, conv_size=384, swin_size=224, is_train=True, use_zscore=True, mean_target=127.3, std_target=41.2):
        self.df = df
        self.img_dir = img_dir
        self.conv_size = conv_size
        self.swin_size = swin_size
        self.is_train = is_train
        self.use_zscore = use_zscore
        self.mean_target = mean_target
        self.std_target = std_target

        self.mean_rgb = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std_rgb = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.df)

    def _normalize(self, img_bgr):
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.tensor(img_rgb, dtype=torch.float32).permute(2, 0, 1) / 255.0
        return (tensor - self.mean_rgb) / self.std_rgb

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = int(row['id'])
        img_path = self.img_dir / f"{img_id}.png"

        img = cv2.imread(str(img_path))
        if img is None:
            img = np.zeros((512, 512, 3), dtype=np.uint8)

        # Áp dụng Augmentation nếu là tập Train
        if self.is_train:
            augmented = train_transform_cv(image=img)
            img = augmented['image']

        img_conv = cv2.resize(img, (self.conv_size, self.conv_size), interpolation=cv2.INTER_AREA)
        img_swin = cv2.resize(img, (self.swin_size, self.swin_size), interpolation=cv2.INTER_AREA)

        tensor_conv = self._normalize(img_conv)
        tensor_swin = self._normalize(img_swin)

        gender = torch.tensor([1.0 if row['male'] else 0.0], dtype=torch.float32)
        raw_boneage = float(row['boneage'])

        if self.use_zscore:
            target = torch.tensor((raw_boneage - self.mean_target) / self.std_target, dtype=torch.float32)
        else:
            target = torch.tensor(raw_boneage, dtype=torch.float32)

        return tensor_conv, tensor_swin, gender, target, torch.tensor(raw_boneage, dtype=torch.float32)

train_dataset = RSNASotaDataset(train_df, IMAGES_DIR, conv_size=CONFIG['IMAGE_SIZE_CONV'], swin_size=CONFIG['IMAGE_SIZE_SWIN'],
                               is_train=True, use_zscore=CONFIG['USE_ZSCORE'], mean_target=BONEAGE_MEAN, std_target=BONEAGE_STD)
val_dataset = RSNASotaDataset(val_df, IMAGES_DIR, conv_size=CONFIG['IMAGE_SIZE_CONV'], swin_size=CONFIG['IMAGE_SIZE_SWIN'],
                             is_train=False, use_zscore=CONFIG['USE_ZSCORE'], mean_target=BONEAGE_MEAN, std_target=BONEAGE_STD)
test_dataset = RSNASotaDataset(test_df, IMAGES_DIR, conv_size=CONFIG['IMAGE_SIZE_CONV'], swin_size=CONFIG['IMAGE_SIZE_SWIN'],
                              is_train=False, use_zscore=CONFIG['USE_ZSCORE'], mean_target=BONEAGE_MEAN, std_target=BONEAGE_STD)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, 
                          num_workers=CONFIG['NUM_WORKERS'], pin_memory=True, drop_last=True,
                          persistent_workers=False)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, 
                        num_workers=CONFIG['NUM_WORKERS'], pin_memory=True,
                        persistent_workers=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, 
                         num_workers=CONFIG['NUM_WORKERS'], pin_memory=True,
                         persistent_workers=False)

print(f"✅ Khởi tạo DataLoader thành công: Train={len(train_loader)} batches | Val={len(val_loader)} batches | Test={len(test_loader)} batches")


---
## 🧠 BƯỚC 4: ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH NHÁNH KÉP SPATIAL TOKEN-LEVEL SOTA
- ConvNeXt-Tiny vi mô (144 spatial tokens) + Swin-Tiny vĩ mô (49 macro tokens).
- Cross-Attention đa đầu: 49 truy vấn vĩ mô tra cứu 144 khóa vi mô sinh ma trận tương quan giải phẫu $49 	imes 144$.
- FiLM Layer điều biến sinh học theo giới tính + Stochastic Depth (DropPath=0.2).


In [ ]:
# ==============================================================================
# BƯỚC 4: ĐỊNH NGHĨA TOÀN BỘ CÁC LỚP KIẾN TRÚC SOTA
# ==============================================================================

class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, metadata_dim=1):
        super().__init__()
        self.generator = nn.Sequential(
            nn.Linear(metadata_dim, 64),
            nn.GELU(),
            nn.Linear(64, feature_dim * 2)
        )
        nn.init.zeros_(self.generator[-1].weight)
        nn.init.zeros_(self.generator[-1].bias)

    def forward(self, x, meta):
        film_params = self.generator(meta)
        gamma, beta = torch.chunk(film_params, 2, dim=-1)
        return x * (1.0 + gamma) + beta


class CrossAttentionFusion(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, 
                                                dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, query_feat, key_feat):
        # query_feat: [B, L_q, D] hoặc [B, D]
        # key_feat: [B, L_kv, D] hoặc [B, D]
        is_q_pooled = query_feat.dim() == 2
        q = query_feat.unsqueeze(1) if is_q_pooled else query_feat
        kv = key_feat.unsqueeze(1) if key_feat.dim() == 2 else key_feat

        attn_out, attn_weights = self.cross_attn(query=q, key=kv, value=kv)
        out = self.norm(q + attn_out)

        if is_q_pooled:
            out = out.squeeze(1)
        return out, attn_weights


class DualBranchBoneAgeNet(nn.Module):
    def __init__(self, convnext_name="convnext_tiny.fb_in22k_ft_in1k_384",
                 swin_name="swin_tiny_patch4_window7_224.ms_in22k_ft_in1k",
                 embed_dim=512, num_heads=8, dropout=0.3, drop_path_rate=0.2):
        super().__init__()

        # 1. Nhánh Vi mô ConvNeXt (Depthwise 7x7 convs)
        self.convnext = timm.create_model(
            convnext_name, pretrained=True, num_classes=0, drop_path_rate=drop_path_rate
        )
        convnext_dim = getattr(self.convnext, "num_features", 768)

        # 2. Nhánh Vĩ mô Swin Transformer (Shifted Windows)
        self.swin = timm.create_model(
            swin_name, pretrained=True, num_classes=0, drop_path_rate=drop_path_rate
        )
        swin_dim = getattr(self.swin, "num_features", 768)

        # 3. Chiếu tuyến tính vào không gian embedding chung
        self.proj_conv = nn.Sequential(nn.Linear(convnext_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())
        self.proj_swin = nn.Sequential(nn.Linear(swin_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())

        # 4. Fusion tương tác không gian thực thụ (Spatial Cross-Attention)
        self.fusion = CrossAttentionFusion(embed_dim=embed_dim, num_heads=num_heads, dropout=0.1)
        self.film = FiLMLayer(feature_dim=embed_dim, metadata_dim=1)

        # 5. Đầu hồi quy phi tuyến sâu
        self.regressor = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def extract_spatial_tokens(self, img_conv, img_swin):
        # ConvNeXt: [B, C, 12, 12] -> norm -> flatten thành 144 spatial tokens [B, 144, C]
        feat_conv = self.convnext.forward_features(img_conv)
        if hasattr(self.convnext, "head") and hasattr(self.convnext.head, "norm"):
            feat_conv = self.convnext.head.norm(feat_conv)
        tokens_conv = feat_conv.flatten(2).permute(0, 2, 1)

        # Swin: [B, 7, 7, C] -> view thành 49 macro tokens [B, 49, C]
        feat_swin = self.swin.forward_features(img_swin)
        if feat_swin.dim() == 4:
            tokens_swin = feat_swin.view(feat_swin.size(0), -1, feat_swin.size(-1))
        else:
            tokens_swin = feat_swin

        return tokens_conv, tokens_swin

    def forward(self, img_conv, img_swin, gender, return_attention=False):
        tokens_c, tokens_s = self.extract_spatial_tokens(img_conv, img_swin)
        f_c = self.proj_conv(tokens_c) # [B, 144, 512]
        f_s = self.proj_swin(tokens_s) # [B, 49, 512]

        # 49 Swin macro queries tra cứu 144 ConvNeXt micro keys/values
        fused_tokens, attn_weights = self.fusion(query_feat=f_s, key_feat=f_c)
        fused = fused_tokens.mean(dim=1) # Pooling toàn cục không gian qua 49 macro queries: [B, 512]

        modulated = self.film(fused, gender)
        out = self.regressor(modulated).squeeze(-1)

        if return_attention:
            return out, attn_weights
        return out

model = DualBranchBoneAgeNet(
    convnext_name=CONFIG['CONVNEXT_MODEL'],
    swin_name=CONFIG['SWIN_MODEL'],
    embed_dim=CONFIG['EMBED_DIM'],
    num_heads=CONFIG['NUM_HEADS'],
    dropout=CONFIG['DROPOUT'],
    drop_path_rate=CONFIG['DROP_PATH_RATE']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Khởi tạo Champion DualBranchBoneAgeNet thành công! Tổng tham số: {total_params/1e6:.2f}M")


---
## 🔄 BƯỚC 5: ĐỘNG CƠ HUẤN LUYỆN SOTA VỚI GIẢI PHÓNG GRADIENT (SMOOTH L1 $\beta=0.1$)
- **Hàm mất mát Smooth L1 với $\beta = 0.1$:** Thu hẹp vùng $L_2$ xuống chỉ 4.1 tháng, duy trì gradient mạnh $\pm 1$ ép MAE xuống $< 5.0$ tháng.
- Hàm `plot_and_save_training_progress()` vẽ và lưu biểu đồ kép chất lượng cao 300 DPI sau mỗi epoch.


In [ ]:
# ==============================================================================
# BƯỚC 5: ĐỘNG CƠ HUẤN LUYỆN SOTA VỚI SMOOTH L1 (BETA=0.1) & LƯU BIỂU ĐỒ 300 DPI
# ==============================================================================
criterion = nn.SmoothL1Loss(beta=CONFIG['SMOOTH_L1_BETA'])
scaler = torch.amp.GradScaler('cuda', enabled=CONFIG['USE_AMP'])

def set_backbone_freeze(model, freeze=True):
    for p in model.convnext.parameters():
        p.requires_grad = not freeze
    for p in model.swin.parameters():
        p.requires_grad = not freeze

set_backbone_freeze(model, freeze=True)
print("🔒 Pha 1: Đóng băng backbones để Warmup heads!")

trainable_params_phase1 = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params_phase1, lr=CONFIG['LR_HEAD_WARMUP'], weight_decay=CONFIG['WEIGHT_DECAY'])

history = {'epoch': [], 'train_loss': [], 'train_mae': [], 'val_loss': [], 'val_mae': [], 'lr': []}
best_val_mae = float('inf')
checkpoint_path = Path(CONFIG['CHECKPOINT_DIR']) / f"{CONFIG['EXPERIMENT']}_fold{CONFIG['FOLD']}_best.pt"

# Hàm suy luận có Test-Time Augmentation (TTA)
def predict_with_tta(model, img_conv, img_swin, gender, use_tta=True):
    if not use_tta:
        with torch.amp.autocast('cuda', enabled=CONFIG['USE_AMP']):
            return model(img_conv, img_swin, gender)
            
    # TTA: Dự đoán ảnh gốc + dự đoán ảnh hơi phóng to/thu nhỏ 103%
    preds = []
    with torch.amp.autocast('cuda', enabled=CONFIG['USE_AMP']):
        p0 = model(img_conv, img_swin, gender)
        preds.append(p0)

        conv_zoomed = torch.nn.functional.interpolate(img_conv, scale_factor=1.03, mode='bilinear', align_corners=False)
        _, _, zh, zw = conv_zoomed.shape
        ch, cw = img_conv.shape[2], img_conv.shape[3]
        dy, dx = (zh - ch) // 2, (zw - cw) // 2
        conv_zoomed = conv_zoomed[:, :, dy:dy+ch, dx:dx+cw]

        swin_zoomed = torch.nn.functional.interpolate(img_swin, scale_factor=1.03, mode='bilinear', align_corners=False)
        _, _, zh, zw = swin_zoomed.shape
        ch, cw = img_swin.shape[2], img_swin.shape[3]
        dy, dx = (zh - ch) // 2, (zw - cw) // 2
        swin_zoomed = swin_zoomed[:, :, dy:dy+ch, dx:dx+cw]

        p1 = model(conv_zoomed, swin_zoomed, gender)
        preds.append(p1)

    return torch.stack(preds, dim=0).mean(dim=0)

# Hàm vẽ và tự động lưu biểu đồ tiến trình huấn luyện 300 DPI
def plot_and_save_training_progress(hist, fold_num, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    axes[0].plot(hist['epoch'], hist['train_loss'], label='Train Loss', color='#1f77b4', lw=2)
    axes[0].plot(hist['epoch'], hist['val_loss'], label='Val Loss', color='#d62728', linestyle='--', lw=2)
    axes[0].set_title(f"Fold {fold_num} Loss (Smooth L1 trên Z-Score, β={CONFIG['SMOOTH_L1_BETA']})", fontweight='bold')
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    b_val = min(hist['val_mae'])
    axes[1].plot(hist['epoch'], hist['train_mae'], label='Train MAE', color='#1f77b4', lw=2)
    axes[1].plot(hist['epoch'], hist['val_mae'], label='Val MAE', color='#d62728', linestyle='--', lw=2)
    axes[1].axhline(b_val, color='#2ca02c', linestyle=':', lw=1.8, label=f"Best Val: {b_val:.2f}m")
    axes[1].set_title(f"Fold {fold_num} Clinical MAE (Tháng tuổi thực tế)", fontweight='bold')
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MAE (tháng)")
    axes[1].legend()

    plt.tight_layout()
    curve_img_path = Path(out_dir) / f"training_curves_fold{fold_num}.png"
    plt.savefig(curve_img_path, dpi=300, bbox_inches='tight')
    plt.show()

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN FOLD {CONFIG['FOLD']}/{CONFIG['NUM_FOLDS'] - 1} ({CONFIG['EPOCHS']} EPOCHS)...")

for epoch in range(1, CONFIG['EPOCHS'] + 1):
    start_time = time.time()

    if epoch == CONFIG['WARMUP_EPOCHS'] + 1:
        print("🔓 Pha 2: MỞ KHÓA TOÀN BỘ BACKBONES ĐỂ FINETUNE END-TO-END!")
        set_backbone_freeze(model, freeze=False)
        optimizer = torch.optim.AdamW([
            {'params': model.convnext.parameters(), 'lr': CONFIG['LR_BACKBONE']},
            {'params': model.swin.parameters(), 'lr': CONFIG['LR_BACKBONE']},
            {'params': model.proj_conv.parameters(), 'lr': CONFIG['LR_HEAD']},
            {'params': model.proj_swin.parameters(), 'lr': CONFIG['LR_HEAD']},
            {'params': model.fusion.parameters(), 'lr': CONFIG['LR_HEAD']},
            {'params': model.film.parameters(), 'lr': CONFIG['LR_HEAD']},
            {'params': model.regressor.parameters(), 'lr': CONFIG['LR_HEAD']},
        ], weight_decay=CONFIG['WEIGHT_DECAY'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CONFIG['EPOCHS'] - CONFIG['WARMUP_EPOCHS'], eta_min=1e-6
        )

    model.train()
    total_train_loss, total_train_mae, train_samples = 0.0, 0.0, 0
    optimizer.zero_grad()

    for step, (img_conv, img_swin, genders, targets_norm, raw_targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch:02d}/{CONFIG['EPOCHS']:02d} [Train]")):
        img_conv = img_conv.to(device, non_blocking=True)
        img_swin = img_swin.to(device, non_blocking=True)
        genders = genders.to(device, non_blocking=True)
        targets_norm = targets_norm.to(device, non_blocking=True)
        raw_targets = raw_targets.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=CONFIG['USE_AMP']):
            preds_norm = model(img_conv, img_swin, genders)
            loss = criterion(preds_norm, targets_norm) / CONFIG['GRADIENT_ACCUMULATION']

        scaler.scale(loss).backward()

        if (step + 1) % CONFIG['GRADIENT_ACCUMULATION'] == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # Giải chuẩn hóa Z-score để tính MAE thực tế theo tháng
        if CONFIG['USE_ZSCORE']:
            preds_months = preds_norm.detach() * BONEAGE_STD + BONEAGE_MEAN
        else:
            preds_months = preds_norm.detach()

        batch_sz = raw_targets.size(0)
        total_train_loss += loss.item() * CONFIG['GRADIENT_ACCUMULATION'] * batch_sz
        total_train_mae += torch.sum(torch.abs(preds_months - raw_targets)).item()
        train_samples += batch_sz

    if epoch > CONFIG['WARMUP_EPOCHS']:
        scheduler.step()

    epoch_train_loss = total_train_loss / train_samples
    epoch_train_mae = total_train_mae / train_samples

    # Đánh giá kiểm định với TTA trên Validation Fold
    model.eval()
    total_val_loss, total_val_mae, val_samples = 0.0, 0.0, 0

    with torch.no_grad():
        for img_conv, img_swin, genders, targets_norm, raw_targets in val_loader:
            img_conv, img_swin = img_conv.to(device), img_swin.to(device)
            genders, targets_norm, raw_targets = genders.to(device), targets_norm.to(device), raw_targets.to(device)

            preds_norm = predict_with_tta(model, img_conv, img_swin, genders, use_tta=CONFIG['USE_TTA'])
            loss = criterion(preds_norm, targets_norm)

            if CONFIG['USE_ZSCORE']:
                preds_months = preds_norm * BONEAGE_STD + BONEAGE_MEAN
            else:
                preds_months = preds_norm

            batch_sz = raw_targets.size(0)
            total_val_loss += loss.item() * batch_sz
            total_val_mae += torch.sum(torch.abs(preds_months - raw_targets)).item()
            val_samples += batch_sz

    epoch_val_loss = total_val_loss / val_samples
    epoch_val_mae = total_val_mae / val_samples
    elapsed = time.time() - start_time
    curr_lr = optimizer.param_groups[-1]['lr']

    history['epoch'].append(epoch)
    history['train_loss'].append(epoch_train_loss)
    history['train_mae'].append(epoch_train_mae)
    history['val_loss'].append(epoch_val_loss)
    history['val_mae'].append(epoch_val_mae)
    history['lr'].append(curr_lr)

    is_best = epoch_val_mae < best_val_mae
    if is_best:
        best_val_mae = epoch_val_mae
        torch.save({
            'epoch': epoch,
            'fold': CONFIG['FOLD'],
            'model_state_dict': model.state_dict(),
            'best_val_mae': best_val_mae,
            'config': CONFIG,
            'boneage_mean': BONEAGE_MEAN,
            'boneage_std': BONEAGE_STD
        }, checkpoint_path)

    print(f"Epoch [{epoch:02d}/{CONFIG['EPOCHS']:02d}] "
          f"Train Loss: {epoch_train_loss:.4f} | Train MAE: {epoch_train_mae:.2f}m | "
          f"Val Loss: {epoch_val_loss:.4f} | Val MAE: {epoch_val_mae:.2f}m | "
          f"Best Val MAE: {best_val_mae:.2f}m {'🌟' if is_best else ''} ({elapsed:.1f}s)", flush=True)
    sys.stdout.flush()

    plot_and_save_training_progress(history, CONFIG['FOLD'], CONFIG['OUTPUT_DIR'])


---
## 📊 BƯỚC 6: XUẤT KẾT QUẢ FOLD HIỆN TẠI (VAL FOLD & TẬP TEST ĐỘC LẬP)
Nạp checkpoint tối ưu của Fold hiện tại và dự đoán trên cả 2 tập:
1. `val_df` (2,018 mẫu) để đánh giá nếp gấp.
2. `test_df` (1,262 mẫu) để sẵn sàng cho bước Ensemble hòa trộn tiếp theo.


In [ ]:
# ==============================================================================
# BƯỚC 6: XUẤT TOÀN BỘ KẾT QUẢ DỰ ĐOÁN CHO FOLD HIỆN TẠI
# ==============================================================================
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Đã nạp thành công checkpoint Fold {CONFIG['FOLD']} từ Epoch {checkpoint['epoch']} với Val MAE: {checkpoint['best_val_mae']:.2f} tháng")

# 1. Dự đoán trên Validation Fold
val_results = []
with torch.no_grad():
    for idx in range(len(val_df)):
        row = val_df.iloc[idx]
        img_conv, img_swin, gender, _, raw_target = val_dataset[idx]
        img_conv = img_conv.unsqueeze(0).to(device)
        img_swin = img_swin.unsqueeze(0).to(device)
        gender = gender.unsqueeze(0).to(device)

        preds_norm = predict_with_tta(model, img_conv, img_swin, gender, use_tta=CONFIG['USE_TTA'])
        pred_months = preds_norm.item() * BONEAGE_STD + BONEAGE_MEAN if CONFIG['USE_ZSCORE'] else preds_norm.item()

        val_results.append({
            'id': int(row['id']),
            'male': bool(row['male']),
            'boneage_true': float(raw_target.item()),
            'boneage_pred': float(pred_months),
            'abs_error': abs(float(pred_months) - float(raw_target.item()))
        })

df_val_preds = pd.DataFrame(val_results)

# 2. Dự đoán trên tập Holdout Test độc lập (1,262 mẫu)
test_results = []
with torch.no_grad():
    for idx in range(len(test_df)):
        row = test_df.iloc[idx]
        img_conv, img_swin, gender, _, raw_target = test_dataset[idx]
        img_conv = img_conv.unsqueeze(0).to(device)
        img_swin = img_swin.unsqueeze(0).to(device)
        gender = gender.unsqueeze(0).to(device)

        preds_norm = predict_with_tta(model, img_conv, img_swin, gender, use_tta=CONFIG['USE_TTA'])
        pred_months = preds_norm.item() * BONEAGE_STD + BONEAGE_MEAN if CONFIG['USE_ZSCORE'] else preds_norm.item()

        test_results.append({
            'id': int(row['id']),
            'male': bool(row['male']),
            'boneage_true': float(raw_target.item()),
            'boneage_pred': float(pred_months),
            'abs_error': abs(float(pred_months) - float(raw_target.item()))
        })

df_test_preds = pd.DataFrame(test_results)

# Thống kê kết quả
test_errors = df_test_preds['abs_error'].values
test_mae = np.mean(test_errors)
test_rmse = np.sqrt(np.mean((df_test_preds['boneage_true'] - df_test_preds['boneage_pred'])**2))
ss_res = np.sum((df_test_preds['boneage_true'] - df_test_preds['boneage_pred'])**2)
ss_tot = np.sum((df_test_preds['boneage_true'] - np.mean(df_test_preds['boneage_true']))**2)
r2_score = 1.0 - (ss_res / ss_tot)

is_puberty_female = (~df_test_preds['male']) & (df_test_preds['boneage_true'] >= 132) & (df_test_preds['boneage_true'] <= 168)
is_puberty_male = (df_test_preds['male']) & (df_test_preds['boneage_true'] >= 144) & (df_test_preds['boneage_true'] <= 180)
puberty_df = df_test_preds[is_puberty_female | is_puberty_male]
puberty_mae = puberty_df['abs_error'].mean() if len(puberty_df) > 0 else 0.0

bootstrap_maes = []
for _ in range(1000):
    sample = np.random.choice(test_errors, size=len(test_errors), replace=True)
    bootstrap_maes.append(np.mean(sample))
ci_lower, ci_upper = np.percentile(bootstrap_maes, [2.5, 97.5])

fold_summary = {
    'experiment': CONFIG['EXPERIMENT'],
    'fold': CONFIG['FOLD'],
    'best_epoch': checkpoint['epoch'],
    'val_mae': float(df_val_preds['abs_error'].mean()),
    'test_mae': float(test_mae),
    'test_mae_ci_95': [float(ci_lower), float(ci_upper)],
    'test_rmse': float(test_rmse),
    'test_r2': float(r2_score),
    'puberty_mae': float(puberty_mae),
}

# Lưu các tệp kết quả
fold_suffix = f"_fold{CONFIG['FOLD']}"
hist_csv = Path(CONFIG['OUTPUT_DIR']) / f"{CONFIG['EXPERIMENT']}{fold_suffix}_history.csv"
val_csv = Path(CONFIG['OUTPUT_DIR']) / f"{CONFIG['EXPERIMENT']}{fold_suffix}_val_predictions.csv"
test_csv = Path(CONFIG['OUTPUT_DIR']) / f"{CONFIG['EXPERIMENT']}{fold_suffix}_test_predictions.csv"
metrics_json = Path(CONFIG['OUTPUT_DIR']) / f"{CONFIG['EXPERIMENT']}{fold_suffix}_metrics.json"

pd.DataFrame(history).to_csv(hist_csv, index=False)
df_val_preds.to_csv(val_csv, index=False)
df_test_preds.to_csv(test_csv, index=False)
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(fold_summary, f, indent=2)

print("=" * 60)
print(f"👑 TỔNG KẾT KẾT QUẢ CHO FOLD {CONFIG['FOLD']}:")
print(f"• Validation Fold MAE: {df_val_preds['abs_error'].mean():.2f} tháng")
print(f"• Holdout Test MAE:    {test_mae:.2f} tháng (95% CI: [{ci_lower:.2f}, {ci_upper:.2f}])")
print(f"• Holdout Test RMSE:   {test_rmse:.2f} tháng")
print(f"• R2 Score:            {r2_score:.4f}")
print(f"• Pubertal Spurt MAE:  {puberty_mae:.2f} tháng (N={len(puberty_df)})")
print(f"💾 File dự đoán test:   {test_csv}")
print(f"💾 File checkpoint:     {checkpoint_path}")
print("=" * 60)


---
## 🏆 BƯỚC 7: TỰ ĐỘNG HÒA TRỘN SOFT-VOTING ENSEMBLE ĐA NẾP GẤP
Tự động quét toàn bộ thư mục `/kaggle/working/` và `/kaggle/input/` để tìm tất cả các file `*_fold*_test_predictions.csv`.
- Nếu có $\ge 2$ folds: Tự động lấy trung bình dự đoán trên 1,262 ca Test độc lập.
- Triệt tiêu phương sai nhiễu ngẫu nhiên, hạ sai số xuống mức kỷ lục và so sánh trực quan qua biểu đồ!


In [ ]:
# ==============================================================================
# BƯỚC 7: TỰ ĐỘNG QUÉT & HÒA TRỘN SOFT-VOTING ENSEMBLE CÁC FOLDS ĐÃ CÓ (AN TOÀN SIÊU TỐC)
# ==============================================================================
detected_fold_files = {}

# 1. Quét trực tiếp tại thư mục đầu ra hiện tại (0.001s)
for p in [Path(CONFIG['OUTPUT_DIR']), Path('/kaggle/working/outputs'), Path('/kaggle/working'), Path('./outputs')]:
    if p.exists():
        for f in p.glob("*_test_predictions.csv"):
            for f_i in range(CONFIG['NUM_FOLDS']):
                if f"fold{f_i}" in f.name and f_i not in detected_fold_files:
                    detected_fold_files[f_i] = f

# 2. Quét nông (Shallow Scan) tại /kaggle/input, BỎ QUA tuyệt đối các thư mục ảnh để không nghẽn FUSE
input_base = Path('/kaggle/input')
if input_base.exists():
    for dataset_dir in input_base.iterdir():
        if dataset_dir.is_dir():
            dname = dataset_dir.name.lower()
            if any(x in dname for x in ['image', 'rsna-bone-age', 'processed', 'data']):
                continue
            candidates = list(dataset_dir.glob("*_test_predictions.csv"))
            out_sub = dataset_dir / "outputs"
            if out_sub.exists():
                candidates += list(out_sub.glob("*_test_predictions.csv"))
            for f in candidates:
                for f_i in range(CONFIG['NUM_FOLDS']):
                    if f"fold{f_i}" in f.name and f_i not in detected_fold_files:
                        detected_fold_files[f_i] = f

print(f"🔍 ĐÃ PHÁT HIỆN {len(detected_fold_files)}/5 FOLDS TRÊN HỆ THỐNG:")
for f_i, path in sorted(detected_fold_files.items()):
    df_f = pd.read_csv(path)
    mae_f = (df_f['boneage_pred'] - df_f['boneage_true']).abs().mean()
    print(f"   • Fold {f_i}: {path.name} | Test MAE đơn lẻ = {mae_f:.2f} tháng")

if len(detected_fold_files) >= 2:
    print("\n🚀 TIẾN HÀNH SOFT-VOTING ENSEMBLE BLENDING TRÊN TẬP TEST ĐỘC LẬP...")
    
    # Nạp và hòa trộn dự đoán
    loaded_dfs = {f_i: pd.read_csv(p) for f_i, p in detected_fold_files.items()}
    ref_df = list(loaded_dfs.values())[0].copy()
    
    ensemble_preds = np.zeros(len(ref_df))
    for f_i, d in loaded_dfs.items():
        ensemble_preds += d['boneage_pred'].values
    ensemble_preds /= len(loaded_dfs)
    
    df_ensemble = ref_df[['id', 'male', 'boneage_true']].copy()
    df_ensemble['boneage_pred'] = ensemble_preds
    df_ensemble['abs_error'] = (df_ensemble['boneage_pred'] - df_ensemble['boneage_true']).abs()
    
    ens_mae = df_ensemble['abs_error'].mean()
    ens_rmse = np.sqrt(np.mean((df_ensemble['boneage_true'] - df_ensemble['boneage_pred'])**2))
    ss_res_e = np.sum((df_ensemble['boneage_true'] - df_ensemble['boneage_pred'])**2)
    ss_tot_e = np.sum((df_ensemble['boneage_true'] - np.mean(df_ensemble['boneage_true']))**2)
    ens_r2 = 1.0 - (ss_res_e / ss_tot_e)
    
    is_pub_f = (~df_ensemble['male']) & (df_ensemble['boneage_true'] >= 132) & (df_ensemble['boneage_true'] <= 168)
    is_pub_m = (df_ensemble['male']) & (df_ensemble['boneage_true'] >= 144) & (df_ensemble['boneage_true'] <= 180)
    pub_ens_df = df_ensemble[is_pub_f | is_pub_m]
    ens_pub_mae = pub_ens_df['abs_error'].mean() if len(pub_ens_df) > 0 else 0.0
    
    # Bootstrap 95% CI cho Ensemble
    boot_ens = []
    e_errs = df_ensemble['abs_error'].values
    for _ in range(1000):
        s = np.random.choice(e_errs, size=len(e_errs), replace=True)
        boot_ens.append(np.mean(s))
    e_ci_l, e_ci_u = np.percentile(boot_ens, [2.5, 97.5])
    
    ens_summary = {
        'num_folds_ensembled': len(detected_fold_files),
        'folds_included': list(sorted(detected_fold_files.keys())),
        'ensemble_test_mae': float(ens_mae),
        'ensemble_mae_ci_95': [float(e_ci_l), float(e_ci_u)],
        'ensemble_rmse': float(ens_rmse),
        'ensemble_r2': float(ens_r2),
        'ensemble_puberty_mae': float(ens_pub_mae),
        'individual_fold_maes': {f"fold_{f_i}": float((loaded_dfs[f_i]['boneage_pred'] - loaded_dfs[f_i]['boneage_true']).abs().mean()) for f_i in detected_fold_files}
    }
    
    ens_csv_path = Path(CONFIG['OUTPUT_DIR']) / "exp5_champion_ensemble_test_predictions.csv"
    ens_json_path = Path(CONFIG['OUTPUT_DIR']) / "exp5_champion_ensemble_metrics.json"
    
    df_ensemble.to_csv(ens_csv_path, index=False)
    with open(ens_json_path, 'w', encoding='utf-8') as f:
        json.dump(ens_summary, f, indent=2)
        
    print("=" * 60)
    print(f"👑 KẾT QUẢ ĐỈNH CAO SOFT-VOTING ENSEMBLE ({len(detected_fold_files)} FOLDS):")
    print(f"• Ensemble Test MAE:   {ens_mae:.2f} tháng (95% CI: [{e_ci_l:.2f}, {e_ci_u:.2f}])")
    print(f"• Ensemble Test RMSE:  {ens_rmse:.2f} tháng")
    print(f"• Ensemble R2 Score:   {ens_r2:.4f}")
    print(f"• Pubertal Spurt MAE:  {ens_pub_mae:.2f} tháng (N={len(pub_ens_df)})")
    print(f"💾 File kết quả Ensemble: {ens_csv_path}")
    print("=" * 60)
    
    # Biểu đồ cột so sánh trực quan từng Fold vs Ensemble
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bar_labels = [f"Fold {f_i}" for f_i in sorted(detected_fold_files.keys())] + ["⭐ 5-Fold Ensemble"]
    bar_values = [ens_summary['individual_fold_maes'][f"fold_{f_i}"] for f_i in sorted(detected_fold_files.keys())] + [ens_mae]
    bar_colors = ['#1f77b4'] * len(detected_fold_files) + ['#2ca02c']
    
    bars = ax.bar(bar_labels, bar_values, color=bar_colors, width=0.55, edgecolor='black', lw=1.2)
    ax.axhline(4.265, color='#d62728', linestyle='--', lw=1.8, label="Kỷ lục Vô địch RSNA 2017 (4.265m)")
    ax.set_ylabel("MAE trên Tập Test Độc Lập (Tháng tuổi)", fontsize=11, fontweight='bold')
    ax.set_title(f"So Sánh Lực Kéo Giảm Sai Số: Từng Fold vs Ensemble ({len(detected_fold_files)} Folds)", fontsize=12, fontweight='bold')
    
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.2f}m", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3),
                    textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.legend(frameon=True, facecolor='white')
    plt.tight_layout()
    ens_bar_path = Path(CONFIG['OUTPUT_DIR']) / "ensemble_comparison_barchart.png"
    plt.savefig(ens_bar_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
else:
    print("\n💡 Mẹo: Hiện tại bạn đã hoàn thành Fold này. Khi chạy các Fold tiếp theo (hoặc gán Output của các lần chạy trước), Cell này sẽ tự động hòa trộn tất cả các Folds để sinh bảng kết quả Ensemble kỷ lục!")
